In [1]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
v1 = model.encode(q1)

In [4]:
v1.shape

(384,)

In [5]:
v2 = model.encode(q2)

In [6]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [7]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
v1.dot(dv)

np.float32(0.323324)

In [9]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [10]:
v2.dot(dv)

np.float32(0.019730432)

In [11]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-07-10 22:09:09--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     738  --.-KB/s    in 0s      

2026-07-10 22:09:09 (40.8 MB/s) - ‘ingest.py’ saved [738/738]



In [12]:
from ingest import load_faq_data
documents = load_faq_data()

In [13]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [14]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [15]:
len(texts)

1375

In [16]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [17]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1375

In [18]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [19]:
import numpy as np
X = np.array(vectors)

In [20]:
scores = X.dot(v1)

In [21]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629412))

In [22]:
documents[2] # my best match was at index 2 the tutorial had it at 553

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [23]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]


In [24]:
scores[top5]

array([0.7629412 , 0.7579371 , 0.7192131 , 0.65363115, 0.56009996],
      dtype=float32)

In [25]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629412
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related

In [26]:
top5

array([  2, 653, 933, 538,   7])

In [27]:
top5 = np.argsort(-scores)[:5]

In [28]:
top5

array([  2, 653, 933, 538,   7])

In [ ]:
from minsearch import VectorSearch
# vindex = vector search index for semantic retrieval over the FAQ embeddings
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [30]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for

In [31]:
!wget https://raw.githubusercontent.com/fdarx/RAG-LLM-zoomcamp/refs/heads/main/01-agentic-rag/code/rag_helper.py

--2026-07-10 22:10:54--  https://raw.githubusercontent.com/fdarx/RAG-LLM-zoomcamp/refs/heads/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2397 (2.3K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.34K  --.-KB/s    in 0s      

2026-07-10 22:10:55 (14.9 MB/s) - ‘rag_helper.py’ saved [2397/2397]



In [32]:
from dotenv import load_dotenv

load_dotenv()
from google import genai
gemini_client = genai.Client()

In [33]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [34]:
from rag_helper import RAGBase

assistant = RAGBase(index, gemini_client)

In [ ]:
assistant.rag('I only found out after the cohort had already kicked off. Is it too late for me to get in?') 
#basic rag query for demonstration of limitation of the text-bassed RAG

"I don't know based on the provided context."

In [46]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [47]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=gemini_client,
)

In [ ]:
query = 'I only found out after the cohort had already kicked off. Is it too late for me to get in?'
vector_assistant.rag(query)
# now running the same query via vector embeddings and we can clearly see the difference here.

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [39]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [40]:
vs_index.fit(vectors, documents)

In [41]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [42]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [52]:
vs_index.close()